In [6]:

import joblib
import pandas as pd

enc = joblib.load("one_hot_encoder.joblib")
model = joblib.load("life_expectancy_model.joblib")


def predict_life_expectancy(
    country_name: str,
    country_code: str,
    year: int,
    features: dict
) -> float:

    # Format categorical inputs exactly as expected by the encoder
    categorical_input = pd.DataFrame([{
        "Country Name": str(country_name),
        "Country Code": str(country_code),
        "Year": f"{year} [YR{year}]"
    }], columns=enc.feature_names_in_)

    # Encode categorical features
    encoded_array = enc.transform(categorical_input)

    encoded_input = pd.DataFrame(
        encoded_array,
        columns=enc.get_feature_names_out(
            categorical_input.columns
        )
    )

    # Prepare numerical features
    features_input = pd.DataFrame([features])

    # Combine all features
    input_data = pd.concat(
        [features_input, encoded_input],
        axis=1
    )

    # Match the model's expected feature order
    expected_features = model.feature_names_in_

    missing = set(expected_features) - set(input_data.columns)
    extra = set(input_data.columns) - set(expected_features)

    if missing:
        raise ValueError(f"Missing model features: {sorted(missing)}")

    if extra:
        raise ValueError(f"Unexpected input features: {sorted(extra)}")

    input_data = input_data[expected_features]

    # Predict life expectancy
    prediction = model.predict(input_data)

    return round(float(prediction[0]), 2)


# Test
#Nigeria Life expectancy prediction as at 2010
features = {
    "index": 100,
    "PM25_AirPollution": 52.61,
    "Population_Density": 182.97,
    "Political_Stability_Index": -2.211,
    "Tertiary_School_Enrollment": 9.57,
    "Health_Expenditure_Per_Capita": 73.01,
    "Urban_Population_Rate": 50.6
}

result = predict_life_expectancy(
    country_name="Nigeria",
    country_code="NGA",
    year=2010,
    features=features
)

print(f"Predicted life expectancy in Nigeria: {result:.2f} years")

#United states prediction as at 2025
features = {
    "index": 1,
    "PM25_AirPollution": 11.4,
    "Population_Density": 98.8,
    "Political_Stability_Index": -0.15,
    "Tertiary_School_Enrollment": 79.36,
    "Health_Expenditure_Per_Capita": 15843.0,
    "Urban_Population_Rate": 82.8
}
result = predict_life_expectancy(
    country_code="USA",
    country_name="United States",
    year=2025,
    features=features
)
print(f"Predicted life expectancy in United States: {result:.2f} years")

Predicted life expectancy in Nigeria: 51.03 years
Predicted life expectancy in United States: 74.19 years


In [2]:
# print("Encoder expects:", enc.feature_names_in_)
#
# for column, categories in zip(enc.feature_names_in_, enc.categories_):
#     print(f"\n{column}")
#     print("Categories:", categories[:10])
#     print("Category dtype:", categories.dtype)